# Calculadora de metas nutricionais diárias

O código nesse *notebook* puxa informações nutricionais de múltiplas fontes para cálculo da quantidade de elementos nutricionais diários listados pela TACO para uma alimentação saudável. Após computar os dados nutricionais necessários, eles serão inseridos num algoritmo de otimização com programação linear, para qualquer uma das características dos alimentos presentes.

- Fontes:
    - UNICAMP/NEPA - Tabela Brasileira de Composição de Alimentos (TACO);
    - NASEM/Health Canada - Dietary Reference Intakes, equations to estimate energy requirement;
    - Health Canada / Food and Nutrition Board - Dietary Reference Intakes tables;
    - National Academies 2019 - Dietary Reference Intakes for Sodium and Potassium;
    - WHO/FAO/UNU 2007 and FAO 2011 protein quality consultation tables.

In [11]:
import csv
from pathlib import Path
import pandas as pd

from functions import (
    calcular_necessidades,
    formatar_numero_brasileiro,
    formatar_numero_exportacao,
    limpar_rotulo,
    otimizar_dieta_lp,
    formatar_quantidade,
    formatar_tabela_exportacao,
    preparar_cobertura_humana,
    preparar_plano_humano,
    preparar_resumo_categorias,
)

from IPython.display import Markdown, display  # pyright: ignore[reportUnknownVariableType]

In [12]:
DATA_DIR = Path("../data")
ID_COL = "Número do Alimento"

alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")

aminoacidos["Triptofano (g)"] = pd.to_numeric(
    aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce",
)

taco_completo = alimentos.merge(
    acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

output_path = DATA_DIR / "taco_completo.csv"
taco_completo.to_csv(output_path, index=False, na_rep="NA")

taco_completo.head()

try:
    tabela_bruta = taco_completo.copy()
except NameError:
    alimentos = pd.read_csv(DATA_DIR / "alimentos.csv")
    acidos_graxos = pd.read_csv(DATA_DIR / "acidos-graxos.csv")
    aminoacidos = pd.read_csv(DATA_DIR / "aminoacidos.csv")
    aminoacidos["Triptofano (g)"] = pd.to_numeric(
        aminoacidos["Triptofano (g)"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    tabela_bruta = alimentos.merge(
        acidos_graxos, on=ID_COL, how="left", suffixes=("", "_acidos_graxos")
    ).merge(aminoacidos, on=ID_COL, how="left", suffixes=("", "_aminoacidos"))

colunas_repetidas = [
    coluna
    for coluna in tabela_bruta.columns
    if coluna.endswith(("_acidos_graxos", "_aminoacidos"))
    and coluna.startswith(("Categoria do alimento", "Descrição dos alimentos"))
]
tabela_limpa = tabela_bruta.drop(columns=colunas_repetidas)

tabela_formatada = tabela_limpa.copy()
colunas_numericas = tabela_formatada.select_dtypes(include="number").columns

for coluna in colunas_numericas:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(formatar_numero_brasileiro)

for coluna in tabela_formatada.columns.difference(colunas_numericas):
    tabela_formatada[coluna] = tabela_formatada[coluna].replace("NA", pd.NA).fillna("")

tabela_formatada.columns = [
    limpar_rotulo(coluna) for coluna in tabela_formatada.columns
]

output_path = DATA_DIR / "taco_completo.csv"
tabela_formatada.to_csv(output_path, sep=";", index=False, quoting=csv.QUOTE_ALL)

tabela_formatada.head()

,Número do Alimento,Categoria do Alimento,Descrição dos Alimentos,Umidade,Energia (kcal),Energia (kJ),Proteína (g),Lipídeos (g),Colesterol (mg),Carboidrato (g),...,Tirosina (g),Valina (g),Arginina (g),Histidina (g),Alanina (g),Ácido Aspártico (g),Ácido Glutâmico (g),Glicina (g),Prolina (g),Serina (g)
0,1,Cereais e derivados,"Arroz, integral, cozido","70,1",124,517,"2,6",1,,"25,8",...,,,,,,,,,,
1,2,Cereais e derivados,"Arroz, integral, cru","12,2",360,1505,"7,3","1,9",,"77,5",...,,,,,,,,,,
2,3,Cereais e derivados,"Arroz, tipo 1, cozido","69,1",128,537,"2,5","0,2",,"28,1",...,,,,,,,,,,
3,4,Cereais e derivados,"Arroz, tipo 1, cru","13,2",358,1497,"7,2","0,3",,"78,8",...,,,,,,,,,,
4,5,Cereais e derivados,"Arroz, tipo 2, cozido","68,7",130,544,"2,6","0,4",,"28,2",...,,,,,,,,,,


In [13]:
necessidades_nutricionais = calcular_necessidades(
    sexo="masculino",
    idade_anos=25,
    altura_m=1.84,
    peso_kg=136,
    colunas_taco=list(tabela_formatada.columns)
)

necessidades_exportacao = necessidades_nutricionais.copy()
for coluna in ["EER usado (kcal/dia)", "Alvo", "Mínimo", "Máximo"]:
    necessidades_exportacao[coluna] = necessidades_exportacao[coluna].map(
        formatar_numero_exportacao
    )

saida_necessidades = Path("../data") / "necessidades_nutricionais_estimadas.csv"
necessidades_exportacao.to_csv(
    saida_necessidades, sep=";", index=False, quoting=csv.QUOTE_ALL
)

necessidades_nutricionais

,Estágio de vida,EER usado (kcal/dia),Nutriente,Colunas TACO usadas,Tipo,Alvo,Mínimo,Máximo,Unidade,Base científica,Observações
0,male_19_30,4098,Energia,Energia (kcal),EER,4098.0,NaN,NaN,kcal/dia,"Equação NASEM 2023 por sexo, idade, altura, pe...",
1,male_19_30,4098,Carboidrato,Carboidrato (g),RDA + AMDR,130.0,461.0,665.8,g/dia,DRI: RDA e 45-65% da energia,
2,male_19_30,4098,Proteína,Proteína (g),RDA por kg + AMDR,108.8,102.4,358.5,g/dia,"0.8 g/kg/dia, com mínimo de referência 56 g/dia",
3,male_19_30,4098,Lipídeos totais,Lipídeos (g),AMDR,NaN,91.1,159.3,g/dia,Percentual de energia vindo de gorduras totais,
4,male_19_30,4098,Fibra Alimentar,Fibra Alimentar (g),AI estimada por energia,57.4,NaN,NaN,g/dia,14 g/1000 kcal; tabela DRI também informa AI p...,AI do estágio de vida na tabela: 38 g/dia
...,...,...,...,...,...,...,...,...,...,...,...
56,male_19_30,4098,Ácido Aspártico (g),Ácido Aspártico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
57,male_19_30,4098,Ácido Glutâmico (g),Ácido Glutâmico (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
58,male_19_30,4098,Glicina (g),Glicina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."
59,male_19_30,4098,Prolina (g),Prolina (g),sem DRI individual,NaN,NaN,NaN,,Sem RDA/AI individual estabelecida para esta c...,"Use como dado de composição alimentar, não com..."


In [14]:
# Linear programming model:
# x[i] = number of 100 g portions of food i.
# Objective: minimize total grams of food while meeting the computed nutrient bounds.
MAX_GRAMAS_POR_ALIMENTO = 500
TOLERANCIA_ENERGIA_ACIMA = 0.05
MIN_GRAMAS_PARA_EXIBIR = 0.1
CATEGORIAS_PERMITIDAS = None  # Example: ["Cereais e derivados", "Carnes e derivados"]
TERMOS_EXCLUIDOS = None  # Example: ["café, pó", "gelatina", "fermento"]

COLUNAS_IDENTIFICACAO = [
    "Número do Alimento",
    "Categoria do Alimento",
    "Descrição dos Alimentos",
]

try:
    tabela_base_lp = tabela_limpa.copy()
except NameError:
    tabela_base_lp = pd.read_csv(DATA_DIR / "taco_completo.csv", sep=";")

try:
    necessidades_lp = necessidades_nutricionais.copy()
except NameError:
    necessidades_lp = pd.read_csv(
        DATA_DIR / "necessidades_nutricionais_estimadas.csv", sep=";"
    )

resumo_otimizacao, dieta_otimizada, cobertura_dieta_otimizada = otimizar_dieta_lp(
    tabela_base=tabela_base_lp,
    necessidades=necessidades_lp,
    max_gramas_por_alimento=MAX_GRAMAS_POR_ALIMENTO,
    tolerancia_energia_acima=TOLERANCIA_ENERGIA_ACIMA,
    categorias_permitidas=CATEGORIAS_PERMITIDAS,
    termos_excluidos=TERMOS_EXCLUIDOS,
    min_gramas_para_exibir=MIN_GRAMAS_PARA_EXIBIR,
)

for caminho, tabela in [
    (DATA_DIR / "dieta_otimizada_lp.csv", dieta_otimizada),
    (DATA_DIR / "cobertura_dieta_otimizada_lp.csv", cobertura_dieta_otimizada),
]:
    tabela_exportacao = tabela.copy()
    for coluna in tabela_exportacao.select_dtypes(include="number").columns:
        tabela_exportacao[coluna] = tabela_exportacao[coluna].map(
            formatar_numero_exportacao
        )
    tabela_exportacao.to_csv(caminho, sep=";", index=False, quoting=csv.QUOTE_ALL)

dieta_otimizada


,Número do Alimento,Categoria do Alimento,Descrição dos Alimentos,Quantidade (g),Porções de 100 g,Energia (kcal) no plano,Proteína (g) no plano,Carboidrato (g) no plano,Lipídeos (g) no plano,Fibra Alimentar (g) no plano,Sódio (mg) no plano
0,501,Produtos açucarados,"Doce, de leite, cremoso",438.030301,4.380303,1340.372720,24.091667,260.628029,26.281818,0.000000,5.256364e+02
1,580,Leguminosas e derivados,"Pé-de-moleque, amendoim",345.453701,3.454537,1737.632118,45.599889,188.963175,96.727036,11.745426,5.527259e+01
2,574,Leguminosas e derivados,"Feijão, roxo, cru",91.066047,0.910660,301.428616,20.216662,54.639628,1.092793,30.780324,9.106605e+00
3,424,Carnes e derivados,Mortadela,90.996558,0.909966,244.780742,10.919587,5.277800,19.655257,0.000000,1.102878e+03
4,253,Frutas e derivados,"Tucumã, cru",62.102717,0.621027,162.709119,1.304157,16.457220,11.861619,7.887045,2.484109e+00
5,46,Cereais e derivados,"Mingau tradicional, pó",24.493193,0.244932,91.359612,0.146959,21.872422,0.097973,0.220439,3.673979e+00
6,515,Miscelâneas,"Gelatina, sabores variados, pó",22.543389,0.225434,85.664877,2.006362,20.108703,0.000002,0.000000,5.297696e+01
7,506,Produtos açucarados,Marmelada,21.689069,0.216891,55.740906,0.086756,15.355861,0.021689,0.889252,2.385798e+00
8,585,Leguminosas e derivados,"Tremoço, cru",9.672102,0.096721,36.850707,3.249826,4.236380,0.996226,3.124089,2.901630e-01
9,594,Nozes e sementes,"Linhaça, semente",7.898671,0.078987,39.098422,1.113713,3.420125,2.551271,2.646055,7.108804e-01


In [15]:
try:
    dieta_base_humana = dieta_otimizada.copy()
except NameError:
    dieta_base_humana = pd.read_csv(DATA_DIR / "dieta_otimizada_lp.csv", sep=";")

try:
    cobertura_base_humana = cobertura_dieta_otimizada.copy()
except NameError:
    cobertura_base_humana = pd.read_csv(
        DATA_DIR / "cobertura_dieta_otimizada_lp.csv", sep=";"
    )

plano_alimentar_legivel = preparar_plano_humano(dieta_base_humana)
resumo_por_categoria = preparar_resumo_categorias(plano_alimentar_legivel)
cobertura_legivel = preparar_cobertura_humana(cobertura_base_humana)

arquivos_legiveis = {
    DATA_DIR / "plano_alimentar_legivel.csv": plano_alimentar_legivel,
    DATA_DIR / "plano_alimentar_resumo_categorias.csv": resumo_por_categoria,
    DATA_DIR / "plano_alimentar_cobertura_legivel.csv": cobertura_legivel,
}

for caminho, tabela in arquivos_legiveis.items():
    formatar_tabela_exportacao(tabela).to_csv(
        caminho, sep=";", index=False, quoting=csv.QUOTE_ALL
    )

total_diario = formatar_quantidade(
    plano_alimentar_legivel["Quantidade diária (g)"].sum()
)
total_semanal = formatar_quantidade(
    plano_alimentar_legivel["Quantidade semanal (g)"].sum()
)
restricoes_ok = int(cobertura_legivel["Status"].eq("OK").sum())
total_restricoes = len(cobertura_legivel)

linhas_markdown = [
    "# Plano alimentar otimizado",
    "",
    f"- Total aproximado: {total_diario} por dia",
    f"- Equivalente semanal: {total_semanal} por semana",
    f"- Alimentos no plano: {len(plano_alimentar_legivel)}",
    f"- Restrições nutricionais atendidas: {restricoes_ok}/{total_restricoes}",
    "",
    "## Alimentos",
    "",
]

for _, linha in plano_alimentar_legivel.iterrows():
    observacao = (
        f" ({linha['Observação prática']})" if linha["Observação prática"] else ""
    )
    linhas_markdown.append(
        f"- {linha['Descrição dos Alimentos']}: {linha['Formato sugerido']}{observacao}"
    )

linhas_markdown.extend(
    [
        "",
        "## Nota",
        "",
        (
            "Este plano minimiza massa total de alimentos, não sabor, "
            "variedade, custo, saciedade ou adequação culinária. Use os "
            "campos CATEGORIAS_PERMITIDAS e TERMOS_EXCLUIDOS na célula "
            "de otimização para deixar o resultado mais parecido com "
            "comida de verdade."
        ),
    ]
)

relatorio_markdown = "\n".join(linhas_markdown)
(DATA_DIR / "plano_alimentar_legivel.md").write_text(
    relatorio_markdown, encoding="utf-8"
)

display(Markdown(relatorio_markdown))
plano_alimentar_legivel

# Plano alimentar otimizado

- Total aproximado: 1120 g por dia
- Equivalente semanal: 7845 g por semana
- Alimentos no plano: 11
- Restrições nutricionais atendidas: 33/33

## Alimentos

- Doce, de leite, cremoso: 440 g por dia (porção diária alta; item denso em açúcar; revisar se quiser um cardápio mais realista)
- Pé-de-moleque, amendoim: 345 g por dia (porção diária alta)
- Feijão, roxo, cru: 91 g por dia (peso da TACO pode mudar após preparo)
- Mortadela: 91 g por dia
- Tucumã, cru: 435 g por semana (~62 g/dia) (peso da TACO pode mudar após preparo)
- Mingau tradicional, pó: 170 g por semana (~24 g/dia) (peso da TACO pode mudar após preparo)
- Gelatina, sabores variados, pó: 160 g por semana (~23 g/dia) (peso da TACO pode mudar após preparo; item denso em açúcar; revisar se quiser um cardápio mais realista)
- Marmelada: 150 g por semana (~22 g/dia) (item denso em açúcar; revisar se quiser um cardápio mais realista)
- Tremoço, cru: 68 g por semana (microquantidade; mais fácil planejar por semana; peso da TACO pode mudar após preparo)
- Linhaça, semente: 55 g por semana (microquantidade; mais fácil planejar por semana; peso da TACO pode mudar após preparo)
- Acerola, crua: 50 g por semana (microquantidade; mais fácil planejar por semana; peso da TACO pode mudar após preparo)

## Nota

Este plano minimiza massa total de alimentos, não sabor, variedade, custo, saciedade ou adequação culinária. Use os campos CATEGORIAS_PERMITIDAS e TERMOS_EXCLUIDOS na célula de otimização para deixar o resultado mais parecido com comida de verdade.

,Descrição dos Alimentos,Categoria do Alimento,Formato sugerido,Quantidade diária (g),Quantidade semanal (g),Quantidade mensal (g),Energia (kcal) no plano,Proteína (g) no plano,Carboidrato (g) no plano,Lipídeos (g) no plano,Fibra Alimentar (g) no plano,Sódio (mg) no plano,Observação prática
0,"Doce, de leite, cremoso",Produtos açucarados,440 g por dia,440.0,3065,13140,1340.0,24.1,260.6,26.3,0.0,526.0,porção diária alta; item denso em açúcar; revi...
1,"Pé-de-moleque, amendoim",Leguminosas e derivados,345 g por dia,345.0,2420,10365,1738.0,45.6,189.0,96.7,11.7,55.0,porção diária alta
2,"Feijão, roxo, cru",Leguminosas e derivados,91 g por dia,91.0,635,2730,301.0,20.2,54.6,1.1,30.8,9.0,peso da TACO pode mudar após preparo
3,Mortadela,Carnes e derivados,91 g por dia,91.0,635,2730,245.0,10.9,5.3,19.7,0.0,1103.0,
4,"Tucumã, cru",Frutas e derivados,435 g por semana (~62 g/dia),62.0,435,1865,163.0,1.3,16.5,11.9,7.9,2.0,peso da TACO pode mudar após preparo
5,"Mingau tradicional, pó",Cereais e derivados,170 g por semana (~24 g/dia),24.0,170,735,91.0,0.1,21.9,0.1,0.2,4.0,peso da TACO pode mudar após preparo
6,"Gelatina, sabores variados, pó",Miscelâneas,160 g por semana (~23 g/dia),23.0,160,675,86.0,2.0,20.1,0.0,0.0,53.0,peso da TACO pode mudar após preparo; item den...
7,Marmelada,Produtos açucarados,150 g por semana (~22 g/dia),22.0,150,650,56.0,0.1,15.4,0.0,0.9,2.0,item denso em açúcar; revisar se quiser um car...
8,"Tremoço, cru",Leguminosas e derivados,68 g por semana,9.5,68,290,37.0,3.2,4.2,1.0,3.1,0.0,microquantidade; mais fácil planejar por seman...
9,"Linhaça, semente",Nozes e sementes,55 g por semana,8.0,55,235,39.0,1.1,3.4,2.6,2.6,1.0,microquantidade; mais fácil planejar por seman...
